In [1]:
import pyrealsense2 as rs
import numpy as np
import cv2
import mediapipe as mp
import warnings
import csv
import os
import time

# Intenta importar la librería de la mano
try:
    from inspire_rh56dftp import InspireHand
    MANO_CONECTADA = True
except ImportError:
    MANO_CONECTADA = False
    print("⚠ No se encontró 'inspire_rh56dftp.py'. El script correrá en modo Simulación.")

# Silenciar advertencia de Protobuf
warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf.symbol_database")

# --- CONFIGURACIÓN DE RED Y BD ---
IP_MANO = "192.168.123.211"
ABIERTO = [1000, 1000, 1000, 1000, 1000, 1000]

DIRECTORIO_GUARDADO = os.path.expanduser("~/Documents/Jupyter/ManoRH56DFTP-2R/Gestos")
DIRECTORIO_RUTINAS = os.path.expanduser("~/Documents/Jupyter/ManoRH56DFTP-2R/Rutinas")
os.makedirs(DIRECTORIO_GUARDADO, exist_ok=True)
os.makedirs(DIRECTORIO_RUTINAS, exist_ok=True)

ARCHIVO_GESTOS_3D = os.path.join(DIRECTORIO_GUARDADO, "gestos_guardados_3d.csv")
ARCHIVO_RUTINAS_3D = os.path.join(DIRECTORIO_RUTINAS, "rutinas_guardadas_3d.csv")
CABECERAS_GESTOS = ["Nombre_Gesto", "Ang_Meñique", "Ang_Anular", "Ang_Corazón", "Ang_Índice", "Ang_Flexión_pulgar", "Ang_Rotación_pulgar"]
CABECERAS_RUTINAS = ["Nombre_Rutina", "Secuencia_Indices", "Secuencia_Nombres"]

# --- FUNCIONES DE CONTROL MANO ---
REG_ANGLE_ACT = 1486
REG_FORCE_SET = 1498  # Registro Modbus para el límite de fuerza configurado
FUERZA_SEGURA = [200, 200, 200, 200, 500, 200]  # Límite de torque conservador para teleoperación

def enviar_angulos(mano, angulos, timeout=5.0, tolerancia=40):
    if not MANO_CONECTADA or mano is None:
        print(f" (Simulación) Enviando ángulos a la mano: {angulos}")
        time.sleep(1)
        return True
        
    mano.send_raw(angulos)
    inicio = time.time()
    while time.time() - inicio < timeout:
        try:
            angulos_act = list(mano.read_holding(REG_ANGLE_ACT, 6))
            if all(abs(act - obj) <= tolerancia for act, obj in zip(angulos_act, angulos)):
                return True
        except Exception: pass 
        time.sleep(0.1)
    print(f" ⚠ Tiempo agotado ({timeout}s) para alcanzar la pose exacta.")
    return False

# --- GESTIÓN DE CSV ---
def cargar_gestos():
    if not os.path.exists(ARCHIVO_GESTOS_3D): return []
    with open(ARCHIVO_GESTOS_3D, mode='r') as file:
        reader = csv.reader(file)
        try: next(reader) 
        except StopIteration: pass
        return [row for row in reader if row]

def cargar_rutinas():
    if not os.path.exists(ARCHIVO_RUTINAS_3D): return []
    with open(ARCHIVO_RUTINAS_3D, mode='r') as file:
        reader = csv.reader(file)
        try: next(reader) 
        except StopIteration: pass
        return [row for row in reader if row and len(row) >= 2]

def exportar_rutina(nombre, indices, gestos_guardados):
    existe = os.path.exists(ARCHIVO_RUTINAS_3D)
    sec_idx = ",".join(str(i) for i in indices)
    sec_nom = " -> ".join([gestos_guardados[i][0] for i in indices])
    with open(ARCHIVO_RUTINAS_3D, mode='a', newline='') as file:
        writer = csv.writer(file)
        if not existe: writer.writerow(CABECERAS_RUTINAS)
        writer.writerow([nombre, sec_idx, sec_nom])
    print(f" ✓ Rutina '{nombre}' guardada.")

# --- MODO 1: CALIBRACIÓN 3D ---
def ejecutar_calibracion(pipeline, align, hands, intrinsics):
    modo = "MÁXIMA"
    muestras_max, muestras_min_d, muestras_min_p = [], [], []
    D_max, D_min = np.zeros(6), np.zeros(6)

    print("\n--- INICIANDO CALIBRACIÓN 3D ---")
    print("Mira la ventana de video y sigue las instrucciones.")
    
    while True:
        frames = pipeline.wait_for_frames()
        aligned_frames = align.process(frames)
        color_frame = aligned_frames.get_color_frame()
        depth_frame = aligned_frames.get_depth_frame()
        if not color_frame or not depth_frame: continue
        
        color_image = np.asanyarray(color_frame.get_data())
        image_rgb = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)
        results = hands.process(image_rgb)
        
        D_actual = np.zeros(6)
        mano_detectada = False
        prof_valida = True

        if results.multi_hand_landmarks and results.multi_handedness:
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                if handedness.classification[0].label == "Left":
                    mano_detectada = True
                    h, w, _ = color_image.shape
                    nodos = [0, 2, 4, 5, 8, 9, 12, 13, 16, 17, 20]
                    puntos_3d = {}
                    pixeles = {}
                    
                    for idx in nodos:
                        lm = hand_landmarks.landmark[idx]
                        cx, cy = max(0, min(int(lm.x * w), w - 1)), max(0, min(int(lm.y * h), h - 1))
                        pixeles[idx] = (cx, cy)
                        z = depth_frame.get_distance(cx, cy)
                        
                        # --- FILTRO DE VECINDAD PARA BORDES ---
                        if z == 0:
                            for dx in [-2, -1, 0, 1, 2]:
                                for dy in [-2, -1, 0, 1, 2]:
                                    nx, ny = max(0, min(cx + dx, w - 1)), max(0, min(cy + dy, h - 1))
                                    z_vecino = depth_frame.get_distance(nx, ny)
                                    if z_vecino > 0:
                                        z = z_vecino
                                        
                                if z > 0: break
                        # --------------------------------------
                        
                        if z > 0:
                            puntos_3d[idx] = np.array(rs.rs2_deproject_pixel_to_point(intrinsics, [cx, cy], z))
                        else:
                            prof_valida = False

                    # --- DIBUJO DEL WIREFRAME EN CALIBRACIÓN ---
                    for idx, (cx, cy) in pixeles.items():
                        cv2.circle(color_image, (cx, cy), 6, (255, 0, 0), cv2.FILLED)
                    
                    if len(pixeles) == 11:
                        cv2.line(color_image, pixeles[0], pixeles[5], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles[0], pixeles[9], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles[0], pixeles[13], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles[0], pixeles[17], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles[0], pixeles[2], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles[2], pixeles[4], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles[5], pixeles[8], (0, 255, 0), 3)   
                        cv2.line(color_image, pixeles[9], pixeles[12], (0, 255, 0), 3)  
                        cv2.line(color_image, pixeles[13], pixeles[16], (0, 255, 0), 3) 
                        cv2.line(color_image, pixeles[17], pixeles[20], (0, 255, 0), 3) 
                        cv2.line(color_image, pixeles[5], pixeles[4], (0, 255, 255), 3) 
                        cv2.line(color_image, pixeles[17], pixeles[4], (255, 0, 255), 3)
                    
                    if prof_valida:
                        D_actual = np.array([
                            np.linalg.norm(puntos_3d[20] - puntos_3d[17]),
                            np.linalg.norm(puntos_3d[16] - puntos_3d[13]),
                            np.linalg.norm(puntos_3d[12] - puntos_3d[9]),
                            np.linalg.norm(puntos_3d[8] - puntos_3d[5]),
                            np.linalg.norm(puntos_3d[4] - puntos_3d[5]),
                            np.linalg.norm(puntos_3d[4] - puntos_3d[17])
                        ])
                    else:
                        cv2.putText(color_image, "Mueve la mano, borde perdiendo profundidad", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

                    if modo == "MÁXIMA":
                        cv2.putText(color_image, f"1. Abierta ({len(muestras_max)}/3) - Presiona 'c'", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                    elif modo == "MÍNIMA_DEDOS":
                        cv2.putText(color_image, f"2. Puno Cerrado ({len(muestras_min_d)}/3) - Presiona 'c'", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                    elif modo == "MÍNIMA_PULGAR":
                        cv2.putText(color_image, f"3. Solo Pulgar ({len(muestras_min_p)}/3) - Presiona 'c'", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)

        cv2.imshow('Calibracion 3D', color_image)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('c') and mano_detectada and prof_valida:
            if modo == "MÁXIMA":
                muestras_max.append(D_actual.copy())
                if len(muestras_max) == 3:
                    D_max = np.mean(np.array(muestras_max), axis=0)
                    print("✓ MÁXIMA capturada. Pasa a Puño cerrado.")
                    modo = "MÍNIMA_DEDOS"
            elif modo == "MÍNIMA_DEDOS":
                muestras_min_d.append(D_actual.copy())
                if len(muestras_min_d) == 3:
                    D_min[0:5] = np.mean(np.array(muestras_min_d), axis=0)[0:5]
                    print("✓ MÍNIMA DEDOS capturada. Pasa a rotación de pulgar.")
                    modo = "MÍNIMA_PULGAR"
            elif modo == "MÍNIMA_PULGAR":
                muestras_min_p.append(D_actual.copy())
                if len(muestras_min_p) == 3:
                    D_min[5] = np.mean(np.array(muestras_min_p), axis=0)[5]
                    print("\n✓ CALIBRACIÓN COMPLETADA CON ÉXITO.")
                    cv2.destroyWindow('Calibracion 3D')
                    return D_max, D_min
        elif key == ord('q'):
            print("⚠ Calibración cancelada. Se usarán valores por defecto.")
            cv2.destroyWindow('Calibracion 3D')
            return None, None

# --- PROGRAMA PRINCIPAL ---
def main():
    mano = None
    if MANO_CONECTADA:
        print(f"Conectando a la mano en {IP_MANO} ...")
        try:
            mano = InspireHand(ip=IP_MANO)
            
            # --- INYECCIÓN DE FUERZA SEGURA ---
            print("Configurando fuerza segura (200) en todos los motores...")
            mano._write_and_ack(REG_FORCE_SET, FUERZA_SEGURA)
            time.sleep(0.1) # Breve pausa para asegurar el cambio en el hardware
            # ----------------------------------
            
            enviar_angulos(mano, ABIERTO)
        except Exception as e:
            print(f"Error conectando a la mano: {e}. Pasando a simulación.")

    # Matriz por defecto 
    D_max = np.array([0.1045, 0.1245, 0.1312, 0.1197, 0.1239, 0.2039])
    D_min = np.array([0.0261, 0.0280, 0.0250, 0.0218, 0.0474, 0.0600]) 
            
    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)

    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
    config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
    align = rs.align(rs.stream.color)
    pipeline.start(config)

    perfil_camara = pipeline.get_active_profile()
    intrinsics = perfil_camara.get_stream(rs.stream.depth).as_video_stream_profile().get_intrinsics()

    # --- MENÚ DE INICIO ---
    print("\n" + "="*40)
    print(" BIENVENIDO AL SISTEMA DE VISIÓN 3D")
    print("="*40)
    print("1. Iniciar cámara de ejecución (Usar calibración predeterminada)")
    print("2. Calibrar mi mano (Recomendado para mayor precisión)")
    opcion = input("Seleccione una opción (1 o 2): ").strip()

    if opcion == '2':
        D_max_nuevo, D_min_nuevo = ejecutar_calibracion(pipeline, align, hands, intrinsics)
        if D_max_nuevo is not None:
            D_max = D_max_nuevo
            D_min = D_min_nuevo
            print("Nuevas matrices en RAM. Transicionando al modo de Ejecución...")

    print("\n=== CÁMARA DE EJECUCIÓN CONTINUA ===")
    print(" 'e' : Iniciar / Pausar Seguimiento en Tiempo Real (Toggle)")
    print(" 's' : Guardar gesto actual en CSV")
    print(" 'r' : Abrir Menú de Rutinas (Consola)")
    print(" 'q' : Salir")

    # Variables para el tracking continuo
    seguimiento_continuo = False
    ultimo_envio = 0
    FRECUENCIA_ENVIO = 0.05 # Tasa de envío (20Hz)
    
    # Variables de filtrado de Zona Muerta
    ultimo_vector_enviado = None
    UMBRAL_MOVIMIENTO = 50 

    try:
        while True:
            frames = pipeline.wait_for_frames()
            aligned_frames = align.process(frames)
            
            color_frame = aligned_frames.get_color_frame()
            depth_frame = aligned_frames.get_depth_frame()
            if not color_frame or not depth_frame: continue
            
            color_image = np.asanyarray(color_frame.get_data())
            image_rgb = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)
            results = hands.process(image_rgb)
            
            vector_modbus_actual = None
            mano_en_pantalla = False
            profundidad_valida = True

            if results.multi_hand_landmarks and results.multi_handedness:
                for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                    if handedness.classification[0].label == "Left":
                        mano_en_pantalla = True
                        h, w, _ = color_image.shape
                        nodos = [0, 2, 4, 5, 8, 9, 12, 13, 16, 17, 20]
                        puntos_3d = {}
                        pixeles_dibujo = {}
                        
                        for idx in nodos:
                            lm = hand_landmarks.landmark[idx]
                            cx, cy = max(0, min(int(lm.x * w), w - 1)), max(0, min(int(lm.y * h), h - 1))
                            pixeles_dibujo[idx] = (cx, cy)
                            
                            z_metros = depth_frame.get_distance(cx, cy)
                            
                            # --- FILTRO DE VECINDAD ---
                            if z_metros == 0:
                                for dx in [-2, -1, 0, 1, 2]:
                                    for dy in [-2, -1, 0, 1, 2]:
                                        nx, ny = max(0, min(cx + dx, w - 1)), max(0, min(cy + dy, h - 1))
                                        z_vecino = depth_frame.get_distance(nx, ny)
                                        if z_vecino > 0:
                                            z_metros = z_vecino
                                    if z_metros > 0: break
                            # --------------------------
                            
                            if z_metros > 0:
                                puntos_3d[idx] = np.array(rs.rs2_deproject_pixel_to_point(intrinsics, [cx, cy], z_metros))
                            else:
                                profundidad_valida = False

                        # Dibujo del Wireframe
                        for idx, (cx, cy) in pixeles_dibujo.items():
                            cv2.circle(color_image, (cx, cy), 6, (255, 0, 0), cv2.FILLED)
                        
                        if len(pixeles_dibujo) == 11:
                            cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[5], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[9], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[13], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[17], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[2], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[2], pixeles_dibujo[4], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[5], pixeles_dibujo[8], (0, 255, 0), 3)   
                            cv2.line(color_image, pixeles_dibujo[9], pixeles_dibujo[12], (0, 255, 0), 3)  
                            cv2.line(color_image, pixeles_dibujo[13], pixeles_dibujo[16], (0, 255, 0), 3) 
                            cv2.line(color_image, pixeles_dibujo[17], pixeles_dibujo[20], (0, 255, 0), 3) 
                            cv2.line(color_image, pixeles_dibujo[5], pixeles_dibujo[4], (0, 255, 255), 3) 
                            cv2.line(color_image, pixeles_dibujo[17], pixeles_dibujo[4], (255, 0, 255), 3)

                        if profundidad_valida:
                            D_actual = np.array([
                                np.linalg.norm(puntos_3d[20] - puntos_3d[17]),
                                np.linalg.norm(puntos_3d[16] - puntos_3d[13]),
                                np.linalg.norm(puntos_3d[12] - puntos_3d[9]),
                                np.linalg.norm(puntos_3d[8] - puntos_3d[5]),
                                np.linalg.norm(puntos_3d[4] - puntos_3d[5]),
                                np.linalg.norm(puntos_3d[4] - puntos_3d[17])
                            ])
                            Apertura_decimal = np.clip((D_actual - D_min) / (D_max - D_min), 0.0, 1.0)
                            vector_modbus_actual = (Apertura_decimal * 1000).astype(int)

            # --- LÓGICA DE SEGUIMIENTO EN TIEMPO REAL CON FILTRO DE ZONA MUERTA ---
            if seguimiento_continuo and mano_en_pantalla and vector_modbus_actual is not None and profundidad_valida:
                if time.time() - ultimo_envio >= FRECUENCIA_ENVIO:
                    if ultimo_vector_enviado is None or np.max(np.abs(vector_modbus_actual - ultimo_vector_enviado)) > UMBRAL_MOVIMIENTO:
                        if MANO_CONECTADA and mano is not None:
                            mano.send_raw(vector_modbus_actual.tolist())
                        
                        ultimo_vector_enviado = vector_modbus_actual.copy()
                        ultimo_envio = time.time()
            # ----------------------------------------------------------------------

            cv2.putText(color_image, "'e' Toggle Tracking | 's' Guardar | 'r' Menú Rutinas | 'q' Salir", (10, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            
            estado_txt = "EN VIVO" if seguimiento_continuo else "PAUSADO"
            color_estado = (0, 0, 255) if seguimiento_continuo else (0, 255, 0) 
            cv2.putText(color_image, f"Teleoperacion: {estado_txt}", (10, 60), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color_estado, 2)
            
            cv2.imshow('Vision 3D - Control de Mano', color_image)

            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('e'):
                seguimiento_continuo = not seguimiento_continuo
                if seguimiento_continuo:
                    ultimo_vector_enviado = None 
                print(f"\n>> MODO TELEOPERACIÓN: {'ACTIVADO' if seguimiento_continuo else 'PAUSADO'}")
            
            elif key == ord('s') and vector_modbus_actual is not None:
                seguimiento_continuo = False 
                print("\n--- GUARDAR NUEVO GESTO ---")
                nombre_gesto = input("Ingrese nombre para este gesto: ").strip()
                if nombre_gesto:
                    existe = os.path.exists(ARCHIVO_GESTOS_3D)
                    with open(ARCHIVO_GESTOS_3D, mode='a', newline='') as file:
                        writer = csv.writer(file)
                        if not existe: writer.writerow(CABECERAS_GESTOS)
                        writer.writerow([nombre_gesto] + vector_modbus_actual.tolist())
                    print(f" ✓ Gesto '{nombre_gesto}' guardado.")
                    
            elif key == ord('r'):
                seguimiento_continuo = False 
                print("\n" + "="*40 + "\n         MENÚ DE RUTINAS 3D\n" + "="*40)
                print("1. Ejecutar un gesto guardado (.csv)")
                print("2. Crear y ejecutar NUEVA Rutina")
                print("3. Importar y ejecutar Rutina guardada")
                print("4. Volver a la cámara")
                opc = input("Seleccione una opción: ")
                gestos = cargar_gestos()
                
                if opc == '1' and gestos:
                    for i, g in enumerate(gestos): print(f"{i}. {g[0]}")
                    try:
                        sel = int(input("Número de gesto (-1 cancelar): "))
                        if 0 <= sel < len(gestos):
                            angs = [int(v) for v in gestos[sel][1:7]]
                            enviar_angulos(mano, angs)
                    except ValueError: pass
                        
                elif opc == '2' and gestos:
                    for i, g in enumerate(gestos): print(f"{i}. {g[0]}")
                    try:
                        indices = [int(x) for x in input("\nSecuencia (ej. 0,2,1): ").split(',') if x.strip()]
                        validos = [i for i in indices if 0 <= i < len(gestos)]
                        if validos:
                            if input("¿Guardar rutina? (s/n): ").lower() == 's':
                                exportar_rutina(input("Nombre: "), validos, gestos)
                            if input("¿Ejecutar ahora? (s/n): ").lower() == 's':
                                for idx in validos:
                                    enviar_angulos(mano, [int(v) for v in gestos[idx][1:7]])
                                    time.sleep(1.0)
                    except ValueError: pass

                elif opc == '3':
                    rutinas = cargar_rutinas()
                    if rutinas:
                        for i, r in enumerate(rutinas): print(f"{i}. {r[0]} | Secuencia: {r[2]}")
                        try:
                            sel = int(input("Número de rutina (-1 cancelar): "))
                            if 0 <= sel < len(rutinas):
                                for idx in [int(x) for x in rutinas[sel][1].split(',') if x.strip()]:
                                    enviar_angulos(mano, [int(v) for v in gestos[idx][1:7]])
                                    time.sleep(1.0)
                        except ValueError: pass
                print("\nRegresando a cámara...")
                
            elif key == ord('q'):
                break

    finally:
        pipeline.stop()
        cv2.destroyAllWindows()
        if mano is not None:
            enviar_angulos(mano, ABIERTO, timeout=2.0)
            mano.close()
        print("Cámara apagada y recursos liberados.")

if __name__ == "__main__":
    main()

Conectando a la mano en 192.168.123.211 ...
Configurando fuerza segura (200) en todos los motores...

 BIENVENIDO AL SISTEMA DE VISIÓN 3D
1. Iniciar cámara de ejecución (Usar calibración predeterminada)
2. Calibrar mi mano (Recomendado para mayor precisión)


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1792762457.381703   14128 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792762457.403359   14128 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Seleccione una opción (1 o 2):  1



=== CÁMARA DE EJECUCIÓN CONTINUA ===
 'e' : Iniciar / Pausar Seguimiento en Tiempo Real (Toggle)
 's' : Guardar gesto actual en CSV
 'r' : Abrir Menú de Rutinas (Consola)
 'q' : Salir

>> MODO TELEOPERACIÓN: ACTIVADO
Cámara apagada y recursos liberados.
